# **AI-Powered Semantic Research Paper Retrieval System**

### This project is an AI-powered research paper retrieval system that helps users find the most relevant research papers based on their search query. It uses semantic search to understand the meaning of the query instead of just matching keywords. The system also generates summaries, extracts important keywords, identifies named entities, and groups papers into topics, making it easier to understand and explore research.

Pipeline of project:-

1. Data Collection
2. NLP Preprocesing
3. Sentence Embedding Generation
4. FAISS Index Construction
5. Semantic Search(Query Retrival)
6. Automatic Summarization
7. Keyword Extraction
8. Named Entity Recognition (NER)
9. Topic Modeling
10. Evaluation


# **Step 1 Data Collection**


In [1]:

!pip install datasets sentence-transformers pandas

In [2]:
from datasets import load_dataset

In [3]:
dataset=load_dataset('CShorten/ML-ArXiv-Papers' , split='train')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [5]:
dataset[0]

{'Unnamed: 0.1': 0,
 'Unnamed: 0': 0.0,
 'title': 'Learning from compressed observations',
 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rat

# **Step 2 NLP Preprocessing**

In [6]:
import pandas as pd

In [7]:
df = pd.DataFrame(dataset)
df

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [8]:
df=df[['title','abstract']]
# the columns named unnamed are not usefull so not taking them

In [9]:
df

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...
117587,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [10]:
df.shape

(117592, 2)

In [11]:
df.isnull().sum()

,0
title,0
abstract,0


In [12]:
df['paper_text'] = df['title']+''+df['abstract']
# taking title and abstract column together

/tmp/ipykernel_21731/2615718260.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['paper_text'] = df['title']+''+df['abstract']


In [13]:
df['paper_text'].head()

,paper_text
0,Learning from compressed observations The pro...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [14]:
type(df[['paper_text']])

pandas.core.frame.DataFrame

In [15]:
print(df['paper_text'].iloc[0])
# USING .iloc pulling out the raw string of just that first entry

Learning from compressed observations  The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regressi

# **Step 3 Sentence Embedding Generation**

In [16]:
!pip install sentence-transformers

In [17]:
!pip install ipywidgets
from sentence_transformers import SentenceTransformer

In [18]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [19]:
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [20]:
sample_text = df['paper_text'].iloc[0]
sample_text

'Learning from compressed observations  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rate functions. The ideas are\nillustrated on the example of nonparame

In [21]:
df.loc[:, 'paper_text'] = df['paper_text'].str.replace('\n', ' ', regex=False)

# regex=False tells pandas to treat \n as a literal string rather than a regular expression pattern making the operation faster and more straightforward

In [22]:
embedding = model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

# before embedding NLP Preprocessing is required

<class 'numpy.ndarray'>
(384,)


In [23]:
embedding[:56]

array([-0.1315641 , -0.00678266, -0.00367612,  0.03265158,  0.11219642,
        0.01227267,  0.09816719, -0.0900523 ,  0.04231161, -0.01977348,
       -0.03308417,  0.07452948,  0.10632038, -0.02060429, -0.02052106,
        0.00169493,  0.07081953,  0.05854454, -0.11231912,  0.02082474,
        0.05692544,  0.0201578 ,  0.0258311 ,  0.0321703 ,  0.10513764,
       -0.09676763,  0.02700802, -0.0234509 , -0.04549678, -0.01013699,
       -0.01794855, -0.04814427,  0.01077652, -0.03759069,  0.01943481,
        0.03715189,  0.02967844,  0.04330941,  0.04373213,  0.03704866,
       -0.00182594,  0.00455183, -0.00799067,  0.03037368, -0.014378  ,
        0.03795147,  0.0595916 , -0.02583356, -0.06521576,  0.05900268,
       -0.02107134,  0.07359422, -0.05720106,  0.00294061,  0.00767515,
       -0.0333416 ], dtype=float32)

In [24]:
sample_embedding=model.encode(df['paper_text'].head(5).to_list())

In [25]:
print(sample_embedding.shape)

(5, 384)


# **Step 4 FAISS Index Construction**

In [26]:
from sklearn.metrics.pairwise import cosine_similarity
# cosine similarity is basicaly used to measure how similar 2 pieces of text are
# if two peices are exact same then 1 , if unrelated then 0

In [27]:
similarity = cosine_similarity(sample_embedding[0].reshape(1 , -1), sample_embedding[0].reshape(1 , -1))
print(similarity)

# hum reshape(1 , -1) isliye use kar rahe h b/c code ko calc karne ke like matrix bana padhta h so in order to tricks the computer into seeing it as a table with 1 row and 384 columns so the function doesn't throw an error.

[[1.0000001]]


In [28]:
similarity = cosine_similarity(sample_embedding[0].reshape(1 , -1), sample_embedding[1].reshape(1 , -1))
print(similarity)

[[0.36625272]]


In [29]:
for i in range(1 , 5):
    sim=similarity = cosine_similarity(sample_embedding[0].reshape(1 , -1), sample_embedding[i].reshape(1 , -1))
    print(similarity)

[[0.36625272]]
[[0.33522844]]
[[0.15505108]]
[[0.37421533]]


In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import numpy as np

# Load the embeddings instantly from your Google Drive
loaded_data = np.load('/content/drive/MyDrive/arxiv_embeddings.npz')
embedding = loaded_data['embeddings']

print("Embeddings loaded instantly! Shape:", embedding.shape)



Embeddings loaded instantly! Shape: (117592, 384)


In [32]:
print(embedding.shape)
print(type(embedding))

(117592, 384)
<class 'numpy.ndarray'>


In [33]:
embedding.dtype

dtype('float32')

FAISS (Facebook AI Similarity Search). This is a super-fast digital library or "index" that allows to search through huge amounts of text data in milliseconds.

In [34]:
!pip install faiss-cpu

In [35]:
import faiss

In [36]:
faiss.normalize_L2(embedding)
# scaling the embedings for easier search

In [37]:
index=faiss.IndexFlatIP(384)
# flat means to store data in its actual form and NO approx or shortcuts
# IP means inner product used to measure similarity , a math term for multiplying numbers together to see how closely they point in the same direction

In [38]:
index.add(embedding)
#adding all the embeddings(117592) to index

In [39]:
print(index.ntotal)

117592


# **Step 5 Semantic Search (Query Retrieval)**

In [40]:
query='deep learning for medical image analysis'
query_embedding = model.encode([query])

query_embedding.shape

(1, 384)

In [41]:
faiss.normalize_L2(query_embedding)

In [42]:
D,I=index.search(query_embedding , 5)
print(D)
print(I)


"""
D, I = we tell FAISS: "Take my 384-number search query, scan the entire database, and give me the top 5 closest papers."

What it returns: It gives you two lists back:
I (Indexes/Row numbers): The exact row ID numbers of the top 5 papers found (e.g., 26063, 59715, etc.).
D (Distances/Scores): The matching scores for those papers (e.g., 0.72, 0.71).
Higher scores mean more similarity.
"""

[[0.7254224  0.72157913 0.7170658  0.71233034 0.70247984]]
[[26063 59715 37161 18030 55928]]


'\nD, I = we tell FAISS: "Take my 384-number search query, scan the entire database, and give me the top 5 closest papers."\n\nWhat it returns: It gives you two lists back:\nI (Indexes/Row numbers): The exact row ID numbers of the top 5 papers found (e.g., 26063, 59715, etc.).\nD (Distances/Scores): The matching scores for those papers (e.g., 0.72, 0.71).\nHigher scores mean more similarity.\n'

In [43]:
print(df.iloc[10466]['title'])
# testing a random paper

A Perspective on Deep Imaging


In [44]:
# Creating a resuable search box
def search_paper(query , k=5):
  query_embedding=model.encode([query]) # does embedding of our query
  faiss.normalize_L2(query_embedding) # scaling
  D,I = index.search(query_embedding,k) #searches in index
  for score,idx in zip(D[0] , I[0]):
    print('Similarity Score' , score)
    print('Title' , df.iloc[idx]['title'])
    print('Abstract' , df.iloc[idx]["abstract"][:500])
    print()

In [45]:
search_paper('deep learning for medical image analysis')

Similarity Score 0.7254224
Title An overview of deep learning in medical imaging focusing on MRI
Abstract   What has happened in machine learning lately, and what does it mean for the
future of medical image analysis? Machine learning has witnessed a tremendous
amount of attention over the last few years. The current boom started around
2009 when so-called deep artificial neural networks began outperforming other
established models on a number of important benchmarks. Deep neural networks
are now the state-of-the-art machine learning models across a variety of areas,
from image analysis to natural l

Similarity Score 0.72157913
Title Medical Imaging with Deep Learning: MIDL 2020 -- Short Paper Track
Abstract   This compendium gathers all the accepted extended abstracts from the Third
International Conference on Medical Imaging with Deep Learning (MIDL 2020),
held in Montreal, Canada, 6-9 July 2020. Note that only accepted extended
abstracts are listed here, the Proceedings of the MIDL 

# **Step 6 Automatic Summarization**

In [46]:
!pip install transformers

In [47]:
import pandas as pd

In [48]:
from transformers import pipeline
summarizer = pipeline(
    'summarization',
    model='facebook/bart-large-cnn',
)

'''
downloading a powerful, pre-trained AI model made by Facebook called BART.
This specific AI has been trained on thousands of news articles and documents,
making it an expert at reading long blocks of text and summarizing them perfectly.
'''

'\ndownloading a powerful, pre-trained AI model made by Facebook called BART.\nThis specific AI has been trained on thousands of news articles and documents,\nmaking it an expert at reading long blocks of text and summarizing them perfectly.\n'

In [49]:
type(summarizer)

transformers.pipelines.text2text_generation.SummarizationPipeline

In [50]:
summary = summarizer(df.iloc[10466]['abstract'], max_length=120 , min_length=40)
# testing it on paper no. 10466 and setting the min and max words of summary to be generated

In [51]:
print(summary)

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]


In [52]:
type(summary[0])

dict

In [53]:
summary[0]['summary_text']
# cleaning the summary and removing the code

'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'

In [54]:
# Combining Search + Summarization
for score,idx in zip(D[0] , I[0]):
    print('Similarity Score' , score)
    print('Title' , df.iloc[idx]['title'])
    print('Abstract' , df.iloc[idx]["abstract"][:500])
    summary = summarizer(df.iloc[idx]['abstract'], max_length=120 , min_length=40)
    print(summary)
    print(summary[0]['summary_text'])
    print()

Similarity Score 0.7254224
Title An overview of deep learning in medical imaging focusing on MRI
Abstract   What has happened in machine learning lately, and what does it mean for the
future of medical image analysis? Machine learning has witnessed a tremendous
amount of attention over the last few years. The current boom started around
2009 when so-called deep artificial neural networks began outperforming other
established models on a number of important benchmarks. Deep neural networks
are now the state-of-the-art machine learning models across a variety of areas,
from image analysis to natural l


Your max_length is set to 120, but your input_length is only 84. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=42)


[{'summary_text': 'Deep neural networks are being used to improve the quality of medical images. The technology is being used in a variety of fields, including medicine and education. It is also being used as a tool to help people understand and understand their health.'}]
Deep neural networks are being used to improve the quality of medical images. The technology is being used in a variety of fields, including medicine and education. It is also being used as a tool to help people understand and understand their health.

Similarity Score 0.72157913
Title Medical Imaging with Deep Learning: MIDL 2020 -- Short Paper Track
Abstract   This compendium gathers all the accepted extended abstracts from the Third
International Conference on Medical Imaging with Deep Learning (MIDL 2020),
held in Montreal, Canada, 6-9 July 2020. Note that only accepted extended
abstracts are listed here, the Proceedings of the MIDL 2020 Full Paper Track
are published in the Proceedings of Machine Learning Resear

Your max_length is set to 120, but your input_length is only 101. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=50)


[{'summary_text': 'This compendium gathers all the accepted extended abstracts from the Third International Conference on Medical Imaging with Deep Learning (MIDL 2020) The conference will be held in Montreal, Canada, 6-9 July 2020.'}]
This compendium gathers all the accepted extended abstracts from the Third International Conference on Medical Imaging with Deep Learning (MIDL 2020) The conference will be held in Montreal, Canada, 6-9 July 2020.

Similarity Score 0.7170658
Title Medical Imaging with Deep Learning: MIDL 2019 -- Extended Abstract Track
Abstract   This compendium gathers all the accepted extended abstracts from the Second
International Conference on Medical Imaging with Deep Learning (MIDL 2019),
held in London, UK, 8-10 July 2019. Note that only accepted extended abstracts
are listed here, the Proceedings of the MIDL 2019 Full Paper Track are
published as Volume 102 of the Proceedings of Machine Learning Research (PMLR)
http://proceedings.mlr.press/v102/.

[{'summary_tex

# **Step 7 KEYWORDS extraction**

In [55]:
def search_and_summarize(query , k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I = index.search(query_embedding,k)
  for score,idx in zip(D[0] , I[0]):
    print('Similarity Score' , score)
    print('Title' , df.iloc[idx]['title'])
    print('title' , df.iloc[idx]["abstract"][:500])
    print()

    summary = summarizer(df.iloc[idx]['abstract'], max_length=120 , min_length=40 , do_sample= False)
    print(summary)
    print(summary[0]['summary_text'])
    print()

In [56]:
search_and_summarize("Deep learning in medical imaging" , k=5)

Your max_length is set to 120, but your input_length is only 84. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=42)


Similarity Score 0.804945
Title Medical Imaging with Deep Learning: MIDL 2020 -- Short Paper Track
title   This compendium gathers all the accepted extended abstracts from the Third
International Conference on Medical Imaging with Deep Learning (MIDL 2020),
held in Montreal, Canada, 6-9 July 2020. Note that only accepted extended
abstracts are listed here, the Proceedings of the MIDL 2020 Full Paper Track
are published in the Proceedings of Machine Learning Research (PMLR).




Your max_length is set to 120, but your input_length is only 101. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=50)


[{'summary_text': 'This compendium gathers all the accepted extended abstracts from the Third International Conference on Medical Imaging with Deep Learning (MIDL 2020) The conference will be held in Montreal, Canada, 6-9 July 2020.'}]
This compendium gathers all the accepted extended abstracts from the Third International Conference on Medical Imaging with Deep Learning (MIDL 2020) The conference will be held in Montreal, Canada, 6-9 July 2020.

Similarity Score 0.78986466
Title Medical Imaging with Deep Learning: MIDL 2019 -- Extended Abstract Track
title   This compendium gathers all the accepted extended abstracts from the Second
International Conference on Medical Imaging with Deep Learning (MIDL 2019),
held in London, UK, 8-10 July 2019. Note that only accepted extended abstracts
are listed here, the Proceedings of the MIDL 2019 Full Paper Track are
published as Volume 102 of the Proceedings of Machine Learning Research (PMLR)
http://proceedings.mlr.press/v102/.


[{'summary_text

In [57]:
pip install keybert==0.8.5

In [58]:
from keybert import KeyBERT

In [59]:
kw_model = KeyBERT()

In [60]:
type(kw_model)

keybert._model.KeyBERT

In [61]:
text=df.iloc[10466]['abstract']
keywords = kw_model.extract_keywords(text)

In [62]:
print(keywords)

[('imaging', 0.4528), ('tomographic', 0.4488), ('reconstruction', 0.3623), ('deep', 0.3003), ('learning', 0.2622)]


In [63]:
print(type(keywords))

<class 'list'>


In [64]:
print(type(keywords [0]))

<class 'tuple'>


In [65]:
keywords=kw_model.extract_keywords(text , keyphrase_ngram_range=(1,3) , stop_words='english')


In [66]:
print(keywords)

[('tomographic imaging deep', 0.6704), ('imaging deep learning', 0.6543), ('learning medical imaging', 0.6041), ('imaging deep', 0.5919), ('medical imaging', 0.5281)]


In [67]:
def search_and_summarize(query , k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I = index.search(query_embedding,k)
  for score,idx in zip(D[0] , I[0]):
    print('Similarity Score' , score)
    print('Title' , df.iloc[idx]['title'])
    print('title' , df.iloc[idx]["abstract"][:500])
    print()

    summary = summarizer(df.iloc[idx]['abstract'], max_length=120 , min_length=40 , do_sample= False)
    print(summary)
    print(summary[0]['summary_text'])
    print()
    keywords=kw_model.extract_keywords(text , keyphrase_ngram_range=(1,3) , stop_words='english')
    print(keywords)
    for k,s in keywords:
      print(k)
    print()


# **Step 8 Name Entity Recognaization**

In [68]:
!pip install scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz (14.8 MB)
  Preparing metadata (setup.py) ... done


In [80]:
import spacy
ner_nlp = spacy.load("en_core_web_sm")

In [81]:
useful_labels = ['ORG', 'GPE', 'DATE', 'EVENT', 'NORP', 'FAC']

def search_and_summarize_with_ner(query, k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D, I = index.search(query_embedding, k)
  for score, idx in zip(D[0], I[0]):
    print('Similarity Score', score)
    print('Title', df.iloc[idx]['title'])
    print('Abstract', df.iloc[idx]['abstract'][:500])
    print()

    summary = summarizer(df.iloc[idx]['abstract'], max_length=120, min_length=40, do_sample=False)
    print(summary[0]['summary_text'])
    print()

    keywords = kw_model.extract_keywords(df.iloc[idx]['abstract'], keyphrase_ngram_range=(1,3), stop_words='english')
    for k, s in keywords:
      print(k)
    print()

    doc = ner_nlp(df.iloc[idx]['abstract'])
    print('Named entities found (organizations, locations, dates, events)')
    for ent in doc.ents:
      if ent.label_ in useful_labels:
        print(ent.text, '-', ent.label_)
    print()



In [82]:
search_and_summarize_with_ner('deep learning for medical image analysis', k=5)

Similarity Score 0.7254224
Title An overview of deep learning in medical imaging focusing on MRI
Abstract   What has happened in machine learning lately, and what does it mean for the
future of medical image analysis? Machine learning has witnessed a tremendous
amount of attention over the last few years. The current boom started around
2009 when so-called deep artificial neural networks began outperforming other
established models on a number of important benchmarks. Deep neural networks
are now the state-of-the-art machine learning models across a variety of areas,
from image analysis to natural l

Deep neural networks are being used to improve the quality of medical images. The technology is being used in a variety of fields, including medicine and education. It is also being used as a tool to help people understand and understand their health.



Your max_length is set to 120, but your input_length is only 84. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=42)


deep learning mri
learning mri
introduction deep learning
learning medical imaging
deep neural

Named entities found (organizations, locations, dates, events)
the last few years - DATE
2009 - DATE

Similarity Score 0.72157913
Title Medical Imaging with Deep Learning: MIDL 2020 -- Short Paper Track
Abstract   This compendium gathers all the accepted extended abstracts from the Third
International Conference on Medical Imaging with Deep Learning (MIDL 2020),
held in Montreal, Canada, 6-9 July 2020. Note that only accepted extended
abstracts are listed here, the Proceedings of the MIDL 2020 Full Paper Track
are published in the Proceedings of Machine Learning Research (PMLR).


This compendium gathers all the accepted extended abstracts from the Third International Conference on Medical Imaging with Deep Learning (MIDL 2020) The conference will be held in Montreal, Canada, 6-9 July 2020.



Your max_length is set to 120, but your input_length is only 101. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=50)


medical imaging deep
imaging deep learning
imaging deep
deep learning midl
conference medical imaging

Named entities found (organizations, locations, dates, events)
the Third
International Conference on Medical Imaging with - ORG
2020 - DATE
Montreal - GPE
Canada - GPE
6-9 July 2020 - DATE
the Proceedings of Machine Learning Research - ORG

Similarity Score 0.7170658
Title Medical Imaging with Deep Learning: MIDL 2019 -- Extended Abstract Track
Abstract   This compendium gathers all the accepted extended abstracts from the Second
International Conference on Medical Imaging with Deep Learning (MIDL 2019),
held in London, UK, 8-10 July 2019. Note that only accepted extended abstracts
are listed here, the Proceedings of the MIDL 2019 Full Paper Track are
published as Volume 102 of the Proceedings of Machine Learning Research (PMLR)
http://proceedings.mlr.press/v102/.


This compendium gathers all the accepted extended abstracts from the Second International Conference on Medical Imaging 

# **Step 9 Topic Modeling (BERTopic)**

Goal: automatically discover the research sub-topics/clusters present in the dataset, using the sentence embeddings we already computed. This lets us tell the user which topic-cluster their search query matched, and how big that cluster is — adding context on top of the existing search + summarization + keyword pipeline.


In [72]:
!pip install bertopic

In [73]:
from bertopic import BERTopic

# min_topic_size raised to 150 to get fewer, broader topics directly from the fit —
topic_model = BERTopic(embedding_model=model, min_topic_size=150, verbose=True)
topics, probs = topic_model.fit_transform(df['paper_text'].tolist(), embeddings=embedding)
df['topic'] = topics

2026-07-05 11:11:50,670 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-05 11:15:16,509 - BERTopic - Dimensionality - Completed ✓
2026-07-05 11:15:16,516 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-05 11:15:50,261 - BERTopic - Cluster - Completed ✓
2026-07-05 11:15:50,311 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-05 11:16:10,079 - BERTopic - Representation - Completed ✓


In [74]:
# HDBSCAN (used inside BERTopic) marks a large fraction of documents as -1 "noise"
# by default. reduce_outliers reassigns every -1 doc to its nearest real topic
# using the embeddings we already computed — so far fewer papers end up unclassified.
new_topics = topic_model.reduce_outliers(
    df['paper_text'].tolist(), topics, strategy="embeddings", embeddings=embedding
)
topic_model.update_topics(df['paper_text'].tolist(), topics=new_topics)
df['topic'] = new_topics

2026-07-05 11:16:17,267 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Check what topics were discovered

In [75]:
topic_info = topic_model.get_topic_info()
topic_info.head(20)

# 'Topic' column: -1 means outlier / no clear topic assigned, ignore those rows
# 'Count' column: number of papers in that topic
# 'Name' column: auto-generated label built from the topic's top keywords

,Topic,Count,Name,Representation,Representative_Docs
0,0,9131,0_reinforcement_policy_rl_learning,"[reinforcement, policy, rl, learning, agent, c...",[Fast Model-based Policy Search for Universal ...
1,1,4777,1_adversarial_attacks_attack_robustness,"[adversarial, attacks, attack, robustness, exa...","[Adversarial Robustness vs Model Compression, ..."
2,2,4087,2_segmentation_images_image_imaging,"[segmentation, images, image, imaging, medical...",[Comprehensive Comparison of Deep Learning Mod...
3,3,3713,3_privacy_federated_fl_private,"[privacy, federated, fl, private, data, client...",[Differential Privacy Meets Federated Learning...
4,4,3640,4_speech_audio_speaker_music,"[speech, audio, speaker, music, recognition, a...",[Audio-visual Speech Enhancement Using Conditi...
5,5,2967,5_regret_bandit_bandits_online,"[regret, bandit, bandits, online, armed, algor...",[Combinatorial Bandits without Total Order for...
6,6,2955,6_graph_node_graphs_gnns,"[graph, node, graphs, gnns, nodes, networks, g...",[Graph Analysis and Graph Pooling in the Spati...
7,7,3057,7_networks_neural_deep_layer,"[networks, neural, deep, layer, network, relu,...",[A Comparative Analysis of the Optimization an...
8,8,2111,8_molecular_protein_drug_molecules,"[molecular, protein, drug, molecules, chemical...",[Learning Geometrically Disentangled Represent...
9,9,2198,9_recommendation_user_items_recommender,"[recommendation, user, items, recommender, ite...",[Intent-Aware Contextual Recommendation System...


In [76]:
topic_label_map = dict(zip(topic_info['Topic'], topic_info['Name']))

Combined search + summarization + keywords + topic context

This is a new function, kept separate from `search_and_summarize` above so nothing already working gets changed. It adds the topic-cluster context on top.

In [77]:
from collections import Counter

def search_and_summarize_with_topic(query, k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)

  D, I = index.search(query_embedding, k)

  # Show the full topic breakdown across the top-k results, not just a single

  retrieved_topics = df.iloc[I[0]]['topic'].tolist()
  topic_counts = Counter(t for t in retrieved_topics if t != -1)

  if topic_counts:
    print("Topics represented in these results:")
    for tid, count in topic_counts.most_common():
      topic_name = topic_label_map.get(tid, "Unknown")
      print(f"  - {topic_name}: {count}/{k} results")
  else:
    print("None of the top results had a clear topic assignment")
  print()

  for score, idx in zip(D[0], I[0]):
    print('Similarity Score', score)
    print('Title', df.iloc[idx]['title'])
    print('Abstract', df.iloc[idx]['abstract'][:500])
    print()

    summary = summarizer(df.iloc[idx]['abstract'], max_length=120, min_length=40, do_sample=False)
    print(summary[0]['summary_text'])
    print()

    keywords = kw_model.extract_keywords(df.iloc[idx]['abstract'], keyphrase_ngram_range=(1,3), stop_words='english')
    print(keywords)
    for kw, s in keywords:
      print(kw)
    print()

In [78]:
search_and_summarize_with_topic('deep learning for medical image analysis', k=5)

Topics represented in these results:
  - 2_segmentation_images_image_imaging: 5/5 results

Similarity Score 0.7254224
Title An overview of deep learning in medical imaging focusing on MRI
Abstract   What has happened in machine learning lately, and what does it mean for the
future of medical image analysis? Machine learning has witnessed a tremendous
amount of attention over the last few years. The current boom started around
2009 when so-called deep artificial neural networks began outperforming other
established models on a number of important benchmarks. Deep neural networks
are now the state-of-the-art machine learning models across a variety of areas,
from image analysis to natural l

Deep neural networks are being used to improve the quality of medical images. The technology is being used in a variety of fields, including medicine and education. It is also being used as a tool to help people understand and understand their health.



Your max_length is set to 120, but your input_length is only 84. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=42)


[('deep learning mri', 0.727), ('learning mri', 0.586), ('introduction deep learning', 0.5595), ('learning medical imaging', 0.5488), ('deep neural', 0.5435)]
deep learning mri
learning mri
introduction deep learning
learning medical imaging
deep neural

Similarity Score 0.72157913
Title Medical Imaging with Deep Learning: MIDL 2020 -- Short Paper Track
Abstract   This compendium gathers all the accepted extended abstracts from the Third
International Conference on Medical Imaging with Deep Learning (MIDL 2020),
held in Montreal, Canada, 6-9 July 2020. Note that only accepted extended
abstracts are listed here, the Proceedings of the MIDL 2020 Full Paper Track
are published in the Proceedings of Machine Learning Research (PMLR).


This compendium gathers all the accepted extended abstracts from the Third International Conference on Medical Imaging with Deep Learning (MIDL 2020) The conference will be held in Montreal, Canada, 6-9 July 2020.



Your max_length is set to 120, but your input_length is only 101. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=50)


[('medical imaging deep', 0.7191), ('imaging deep learning', 0.6699), ('imaging deep', 0.5934), ('deep learning midl', 0.5715), ('conference medical imaging', 0.5242)]
medical imaging deep
imaging deep learning
imaging deep
deep learning midl
conference medical imaging

Similarity Score 0.7170658
Title Medical Imaging with Deep Learning: MIDL 2019 -- Extended Abstract Track
Abstract   This compendium gathers all the accepted extended abstracts from the Second
International Conference on Medical Imaging with Deep Learning (MIDL 2019),
held in London, UK, 8-10 July 2019. Note that only accepted extended abstracts
are listed here, the Proceedings of the MIDL 2019 Full Paper Track are
published as Volume 102 of the Proceedings of Machine Learning Research (PMLR)
http://proceedings.mlr.press/v102/.


This compendium gathers all the accepted extended abstracts from the Second International Conference on Medical Imaging with Deep Learning (MIDL 2019) The conference will be held in London, UK,

# **Step 10 Evaluation Metric**


In [79]:
import random

def evaluate_retrieval(sample_size=200, k=5):
  random.seed(42)
  sample_idx = random.sample(range(len(df)), sample_size)

  hits = 0
  for idx in sample_idx:
    query = df.iloc[idx]['title']
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)
    if idx in I[0]:
      hits = hits + 1

  precision = hits / sample_size
  print('Sample size', sample_size)
  print('k', k)
  print('Precision@k', precision)
  print(hits, 'out of', sample_size, 'papers were correctly retrieved using their own title as query')

evaluate_retrieval(sample_size=200, k=5)

Sample size 200
k 5
Precision@k 0.95
190 out of 200 papers were correctly retrieved using their own title as query
